# 📈 สนามเล่น: AI ตรวจความผิดปกติ (บทที่ 5)

ไม่ต้องมีบอร์ด ไม่ต้องมี GPU ไม่ต้องติดตั้งอะไร กด **Runtime → Run all** ได้เลย
โค้ดในเซลล์ `detect_anomalies` **เป็นอันเดียวกับ** `gateway/mqtt_collector/ai_helpers.py` ที่ใช้จริง

สิ่งที่จะได้ลอง
1. สร้างข้อมูลเซนเซอร์จำลอง แล้วดู z-score
2. ปรับ **ความแกว่ง / ขนาดการกระโดด / เกณฑ์ z_limit** แล้วหาจุดที่ AI เริ่มเตือน
3. เห็นกับตาว่า **เตือนผิด (false alarm)** กับ **ตรวจไม่เจอ (miss)** เกิดเมื่อไหร่
4. ใส่ข้อมูลจริงจากบอร์ด (CSV) ของตัวเอง

In [ ]:
import statistics, random
import matplotlib.pyplot as plt

MIN_STD = {1: 0.2, 2: 0.2, 3: 0.2, 4: 0.5}

def detect_anomalies(series, z_limit=3.0, min_points=20):
    """เหมือนใน ai_helpers.py: series = {node: [[ts, value], ...]}"""
    results = []
    for node, points in series.items():
        values = [p[1] for p in points]
        if len(values) < min_points:
            results.append({"node": node, "status": "not_enough_data", "points": len(values)}); continue
        split = max(min_points // 2, int(len(values) * 0.8))
        baseline, recent = values[:split], values[split:]
        mean_b = statistics.fmean(baseline)
        std_b = max(statistics.pstdev(baseline), MIN_STD.get(node, 0.2))
        z = (statistics.fmean(recent) - mean_b) / std_b
        results.append({"node": node, "status": "anomaly" if abs(z) > z_limit else "normal",
                        "z": round(z, 2), "baseline_mean": round(mean_b, 2),
                        "recent_mean": round(statistics.fmean(recent), 2), "std": round(std_b, 3)})
    return results
print("พร้อมแล้ว")

In [ ]:
#@title ลองเล่น: สร้างข้อมูลจำลองแล้วดูว่า AI เตือนไหม
NOISE = 0.2  #@param {type:"slider", min:0, max:1, step:0.05}
JUMP = 1.5  #@param {type:"slider", min:-5, max:5, step:0.1}
EVENT = "jump"  #@param ["jump", "drift", "none"]
Z_LIMIT = 3.0  #@param {type:"slider", min:1, max:6, step:0.1}
SEED = 7  #@param {type:"integer"}

random.seed(SEED)
N = 60
vals = []
for i in range(N):
    shift = JUMP if (EVENT == "jump" and i >= int(N * 0.8)) else JUMP * i / (N - 1) if EVENT == "drift" else 0
    vals.append(30 + shift + random.gauss(0, NOISE))
res = detect_anomalies({3: [[i, v] for i, v in enumerate(vals)]}, Z_LIMIT)[0]
print(res)

split = int(N * 0.8)
plt.figure(figsize=(9, 3.5))
plt.axhspan(res["baseline_mean"] - Z_LIMIT * res["std"], res["baseline_mean"] + Z_LIMIT * res["std"], alpha=.15, color="g", label="ช่วงปกติ")
plt.plot(vals, ".-", color="k", lw=.6)
plt.plot(range(split, N), vals[split:], "o", color="tab:orange" if res["status"] == "anomaly" else "g")
plt.axvline(split - .5, color="gray", ls=":")
plt.title(("⚠ ANOMALY" if res["status"] == "anomaly" else "normal") + f"   z = {res['z']}")
plt.xlabel("ลำดับการอ่านค่า"); plt.ylabel("°C"); plt.legend(); plt.show()

## 🧪 โจทย์ให้ลอง (เปลี่ยนค่าในเซลล์ด้านบนแล้วกด ▶ ใหม่)
1. ตั้ง `EVENT = "none"` แล้วลด `Z_LIMIT` เหลือ 1.0 แล้วเปลี่ยน `SEED` หลายๆ ค่า เกิด **เตือนผิด** บ่อยแค่ไหน
2. ตั้ง `EVENT = "jump"`, `JUMP = 0.5`, `Z_LIMIT = 3` AI เตือนไหม ต้อง `JUMP` เท่าไหร่ถึงเตือน
3. ตั้ง `EVENT = "drift"`, `JUMP = 4` อุณหภูมิเพิ่มถึง 4 °C แต่ AI เตือนไหม **ทำไมจึงไม่เตือน** (คิดดูว่าช่วงอ้างอิงเป็นอะไร)
4. เพิ่ม `NOISE` เป็น 0.8 แล้วหา `JUMP` ที่ทำให้เตือน สัญญาณรบกวนทำให้จับความผิดปกติยากขึ้นอย่างไร

## ใช้ข้อมูลจริงจากบอร์ดของคุณ
ที่เครื่องที่รัน gateway ดึงข้อมูลเป็นไฟล์ แล้วอัปโหลดมาที่ Colab (ไอคอนโฟลเดอร์ ด้านซ้าย)

```bash
curl 'localhost:5001/api/summary?minutes=60' > summary.json
```

In [ ]:
#@title (ไม่บังคับ) วิเคราะห์ summary.json ที่อัปโหลดมา
import json, os
if os.path.exists("summary.json"):
    data = json.load(open("summary.json"))
    series = data.get("series") or data.get("history") or {}
    series = {int(k): v for k, v in series.items()}
    for r in detect_anomalies(series): print(r)
else:
    print("ยังไม่มีไฟล์ summary.json: ข้ามเซลล์นี้ได้")